In [2]:
import numpy as np
import pandas as pd
import os
import random
from tqdm import tqdm
from collections import defaultdict

In [3]:
POI_DIR    = "poi_clusters"
IDS_DIR    = "ids"
CTX_DIR    = "context_embeddings"
OUT_DIR    = "quintuplet_tuples"

MAX_TUPLES_PER_POI = 300
MIN_PHOTOS_PER_POI = 5

USA_LAT    = (24.0, 49.5)
EUROPE_LAT = (35.0, 72.0)
USA_LON    = (-125.0, -66.5)
EUROPE_LON = (-10.0, 40.0)

os.makedirs(OUT_DIR, exist_ok=True)
print("Config ready.")
print(f"  Max tuples per POI : {MAX_TUPLES_PER_POI}")
print(f"  Min photos per POI : {MIN_PHOTOS_PER_POI}")

Config ready.
  Max tuples per POI : 300
  Min photos per POI : 5


In [4]:
print("Loading lat/lon for geo-distance fallback...")

df_coords = pd.read_csv(
    "usa_europe_geotagged.csv",
    usecols=['photoid', 'latitude', 'longitude'],
    dtype={'photoid': np.int64, 'latitude': np.float32, 'longitude': np.float32}
)

df_coords = df_coords.dropna(subset=['latitude', 'longitude'])

photoid_to_lat = dict(zip(df_coords['photoid'].values, df_coords['latitude'].values))
photoid_to_lon = dict(zip(df_coords['photoid'].values, df_coords['longitude'].values))

def is_usa(pid):
    lat = photoid_to_lat.get(int(pid), None)
    lon = photoid_to_lon.get(int(pid), None)
    if lat is None: return False
    return USA_LAT[0] <= lat <= USA_LAT[1] and USA_LON[0] <= lon <= USA_LON[1]

def is_europe(pid):
    lat = photoid_to_lat.get(int(pid), None)
    lon = photoid_to_lon.get(int(pid), None)
    if lat is None: return False
    return EUROPE_LAT[0] <= lat <= EUROPE_LAT[1] and EUROPE_LON[0] <= lon <= EUROPE_LON[1]

def different_region(anchor_pid, candidate_pid):
    return is_usa(anchor_pid) != is_usa(candidate_pid) or is_europe(anchor_pid) != is_europe(candidate_pid)

print(f"Loaded coords for : {len(photoid_to_lat):,} photos")
print("Geo-distance functions ready.")

Loading lat/lon for geo-distance fallback...
Loaded coords for : 7,672,417 photos
Geo-distance functions ready.


In [5]:
photoids  = np.load(os.path.join(POI_DIR, "photoid_to_poi.npy"))
poi_labels = np.load(os.path.join(POI_DIR, "poi_labels.npy"))

# remove noise points (label == -1)
valid_mask = poi_labels != -1
photoids   = photoids[valid_mask]
poi_labels = poi_labels[valid_mask]

print(f"Total photos after removing noise : {len(photoids):,}")
print(f"Unique POIs                        : {len(np.unique(poi_labels)):,}")
print(f"Sample photoids                    : {photoids[:5]}")
print(f"Sample poi labels                  : {poi_labels[:5]}")

Total photos after removing noise : 6,921,827
Unique POIs                        : 239,046
Sample photoids                    : [29872 29873 29874 29875 30431]
Sample poi labels                  : [0 0 0 0 1]


In [6]:
ctx_id_files = sorted([f for f in os.listdir(CTX_DIR) if f.startswith("context_ids_chunk_")])
ctx_feat_files = sorted([f for f in os.listdir(CTX_DIR) if f.startswith("context_chunk_")])

all_ctx_photoids = []
all_uid_ints     = []

for id_file, feat_file in tqdm(zip(ctx_id_files, ctx_feat_files), total=len(ctx_id_files)):
    pids     = np.load(os.path.join(CTX_DIR, id_file))
    features = np.load(os.path.join(CTX_DIR, feat_file))
    uid_ints = features[:, 0].round().astype(np.int32) 

    all_ctx_photoids.append(pids)
    all_uid_ints.append(uid_ints)

all_ctx_photoids = np.concatenate(all_ctx_photoids)
all_uid_ints     = np.concatenate(all_uid_ints)

photoid_to_uid = dict(zip(all_ctx_photoids.tolist(), all_uid_ints.tolist()))

print(f"Loaded uid mappings for : {len(photoid_to_uid):,} photos")
print(f"Sample uid_int values   : {all_uid_ints[:5]}")

100%|████████████████████████████████████████████████████████████████████████████████| 768/768 [00:28<00:00, 27.18it/s]


Loaded uid mappings for : 7,672,417 photos
Sample uid_int values   : [31070 31070 31070 31070 31079]


In [7]:
# group photoids by poi_id
poi_to_photos = defaultdict(list)

for pid, poi_id in tqdm(zip(photoids, poi_labels), total=len(photoids)):
    poi_to_photos[poi_id].append(pid)

# filter out POIs with less than MIN_PHOTOS_PER_POI
poi_to_photos = {
    poi_id: photos
    for poi_id, photos in poi_to_photos.items()
    if len(photos) >= MIN_PHOTOS_PER_POI
}

print(f"POIs before filter : {len(np.unique(poi_labels)):,}")
print(f"POIs after filter  : {len(poi_to_photos):,}")
print(f"Removed POIs       : {len(np.unique(poi_labels)) - len(poi_to_photos):,}")

# check average photos per POI
sizes = [len(v) for v in poi_to_photos.values()]
print(f"Avg photos per POI : {np.mean(sizes):.1f}")
print(f"Max photos per POI : {np.max(sizes):,}")
print(f"Min photos per POI : {np.min(sizes):,}")

100%|███████████████████████████████████████████████████████████████████| 6921827/6921827 [00:03<00:00, 1777581.50it/s]


POIs before filter : 239,046
POIs after filter  : 145,933
Removed POIs       : 93,113
Avg photos per POI : 45.3
Max photos per POI : 291,178
Min photos per POI : 5


In [8]:
# for each user, track which POIs they have photos at
user_to_pois = defaultdict(set)

for pid, poi_id in tqdm(zip(photoids, poi_labels), total=len(photoids)):
    if poi_id in poi_to_photos:  # only valid POIs
        uid = photoid_to_uid.get(int(pid), -1)
        if uid != -1:
            user_to_pois[uid].add(poi_id)

print(f"Total users with valid POIs : {len(user_to_pois):,}")

# check distribution
poi_counts = [len(v) for v in user_to_pois.values()]
print(f"Avg POIs per user           : {np.mean(poi_counts):.1f}")
print(f"Max POIs per user           : {np.max(poi_counts):,}")
print(f"Min POIs per user           : {np.min(poi_counts):,}")

100%|████████████████████████████████████████████████████████████████████| 6921827/6921827 [00:07<00:00, 942084.33it/s]

Total users with valid POIs : 72,380
Avg POIs per user           : 8.0
Max POIs per user           : 1,348
Min POIs per user           : 1


In [9]:
print("Prebuilding fast lookup structures...")

# for each POI, group photos by uid
poi_uid_photos = {}

for poi_id, photos in tqdm(poi_to_photos.items()):
    uid_groups = defaultdict(list)
    for pid in photos:
        uid = photoid_to_uid.get(int(pid), -1)
        if uid != -1:
            uid_groups[uid].append(pid)
    # only keep POIs with at least 2 different users
    if len(uid_groups) >= 2:
        poi_uid_photos[poi_id] = dict(uid_groups)

print(f"POIs with 2+ users : {len(poi_uid_photos):,}")

all_poi_ids = list(poi_uid_photos.keys())

Prebuilding fast lookup structures...


100%|███████████████████████████████████████████████████████████████████████| 145933/145933 [00:06<00:00, 21303.42it/s]

POIs with 2+ users : 61,491


In [10]:
print("Mining quintuplet tuples (300 cap + geo fallback)...")

all_tuples = []

# build per-user photo list for geo fallback
user_to_photos = defaultdict(list)
for pid, poi_id in zip(photoids, poi_labels):
    uid = photoid_to_uid.get(int(pid), -1)
    if uid != -1:
        user_to_photos[uid].append(pid)

for poi_id in tqdm(all_poi_ids):

    uid_groups  = poi_uid_photos[poi_id]
    uid_list    = list(uid_groups.keys())
    tuples_made = 0

    random.shuffle(uid_list)

    for anchor_uid in uid_list:

        if tuples_made >= MAX_TUPLES_PER_POI:
            break

        anchor_photos   = uid_groups[anchor_uid]
        user_other_pois = list(user_to_pois[anchor_uid] - {poi_id})
        diff_uids       = [u for u in uid_list if u != anchor_uid]

        for anchor in anchor_photos:

            if tuples_made >= MAX_TUPLES_PER_POI:
                break

            # P1 — same user, same POI
            p1_pool = [p for p in anchor_photos if p != anchor]
            p1      = random.choice(p1_pool) if p1_pool else anchor

            # P2 — different user, same POI
            p2_uid = random.choice(diff_uids)
            p2     = random.choice(uid_groups[p2_uid])

            # N1 — same user, different POI (option 1)
            # fallback — same user, different region (option 2)
            if len(user_other_pois) > 0:
                n1_poi = random.choice(user_other_pois)
                n1     = random.choice(poi_to_photos[n1_poi])
            else:
                # geo fallback — find a photo from same user in different region
                user_photos  = user_to_photos[anchor_uid]
                diff_region  = [p for p in user_photos if different_region(anchor, p)]
                if len(diff_region) > 0:
                    n1 = random.choice(diff_region)
                else:
                    # last resort — any other photo from same user
                    other_photos = [p for p in user_photos if p != anchor]
                    if len(other_photos) == 0:
                        continue
                    n1 = random.choice(other_photos)

            # N2 — different user, different POI
            n2_poi = random.choice(all_poi_ids)
            while n2_poi == poi_id:
                n2_poi = random.choice(all_poi_ids)
            n2 = random.choice(poi_to_photos[n2_poi])

            all_tuples.append((anchor, p1, p2, n1, n2))
            tuples_made += 1

print(f"\nTotal tuples mined : {len(all_tuples):,}")
print(f"Expected max       : {len(all_poi_ids) * MAX_TUPLES_PER_POI:,}")

Mining quintuplet tuples (300 cap + geo fallback)...


100%|███████████████████████████████████████████████████████████████████████████| 61491/61491 [03:38<00:00, 280.89it/s]


Total tuples mined : 2,034,723
Expected max       : 18,447,300


In [11]:
print("Saving quintuplet tuples...")

tuples_array = np.array(all_tuples, dtype=np.int64)

np.save(os.path.join(OUT_DIR, "quintuplet_tuples.npy"), tuples_array)

print(f"Shape        : {tuples_array.shape}")
print(f"Dtype        : {tuples_array.dtype}")
print(f"File size    : {tuples_array.nbytes / 1e6:.1f} MB")
print(f"Sample tuple : {tuples_array[0]}")
print(f"Columns      : [anchor, p1, p2, n1, n2]")
print(f"Saved → {OUT_DIR}/quintuplet_tuples.npy")

Saving quintuplet tuples...
Shape        : (2034723, 5)
Dtype        : int64
File size    : 81.4 MB
Sample tuple : [ 309237268  312611979    5020185 7965822322 1684143996]
Columns      : [anchor, p1, p2, n1, n2]
Saved → quintuplet_tuples/quintuplet_tuples.npy
